# Real precipitation — styled plot **and animation**

A real **MSWEP daily precipitation** series (mm/day) over the Rhine basin, rendered with cleopatra's
`total_precipitation` style. A fixed 0…max scale keeps wet and dry days comparable across the animation.
A second animation shows the matching **river discharge** (`Qtot`) series.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import cleopatra
from cleopatra.array_glyph import ArrayGlyph, FrameLabel
from cleopatra.animation import embed_gif
print("cleopatra", cleopatra.__version__)

DATA = Path("../data")   # kernel CWD is the notebook's own folder

def load_stack(name):
    """Load an .npz holding a (time, rows, cols) stack, its extent, and date labels."""
    z = np.load(DATA / name, allow_pickle=True)
    return z["stack"].astype(float), [float(v) for v in z["extent"]], list(z["labels"])

precip, p_ext, p_labels = load_stack("mswep_precip.npz")
print("MSWEP precip stack:", precip.shape, "|", p_labels[0], "->", p_labels[-1],
      "| max", round(float(np.nanmax(precip)), 1), "mm/day")

## The wettest day

One frame with the `total_precipitation` style, drawn with `ArrayGlyph.plot`.

In [ ]:
pmax = float(np.nanmax(precip))
day = int(np.nanargmax([np.nansum(f) for f in precip]))
pw, pe, ps, pn = p_ext
glyph = ArrayGlyph(precip[day], extent=[pw, ps, pe, pn])
glyph.plot(style="total_precipitation", vmin=0, vmax=pmax, full_bleed=True)   # fills the frame, no white margin
plt.show()

## Animated — the daily precipitation series

Each frame is the same `total_precipitation` style on that day, on a fixed 0…max scale.

In [ ]:
# figure sized to the data box so the equal-aspect map fills it (no letterbox)
pw, pe, ps, pn = p_ext
fw, fh = pe - pw, pn - ps
fs = (8, 8 * fh / fw) if fw >= fh else (8 * fw / fh, 8)
glyph = ArrayGlyph(precip, extent=[pw, ps, pe, pn], figsize=fs)
anim = glyph.animate(p_labels, style="total_precipitation", vmin=0, vmax=pmax,
                     title="MSWEP daily precipitation",
                     frame_label=FrameLabel(location=[pw + 0.03 * (pe - pw),
                                                      ps + 0.05 * (pn - ps)]),
                     interval=500)
plt.close(glyph.fig)
embed_gif(anim, fps=2)

## Bonus — river discharge (`Qtot`) series

The matching Rhine runoff series (10 days), rendered with the `flow_accumulation` style (symmetric-log,
value-linked opacity) so the wet channel network stands out.

In [ ]:
q, q_ext, q_labels = load_stack("rhine_discharge.npz")
qmax = float(np.nanmax(q))
qw, qe, qs, qn = q_ext
fw, fh = qe - qw, qn - qs
fs = (8, 8 * fh / fw) if fw >= fh else (8 * fw / fh, 8)
glyph = ArrayGlyph(q, extent=[qw, qs, qe, qn], figsize=fs)
anim2 = glyph.animate(q_labels, style="flow_accumulation", vmin=0, vmax=qmax,
                      title="Rhine discharge Qtot",
                      frame_label=FrameLabel(location=[qw + 0.03 * (qe - qw),
                                                       qs + 0.05 * (qn - qs)]),
                      interval=400)
plt.close(glyph.fig)
embed_gif(anim2, fps=3)